# Federated Learning 

> Splits the diabetes dataset into N independent client datasets, simulating hospitals that cannot share raw patient data. Only the model is shared — never the data.

---

## Strategy

Partitioning is done at the **patient level**: all encounters from the same patient go to the same client, so no patient can appear in two hospitals simultaneously. This prevents data leakage across clients.

| Stage | What it does |
|---|---|
| **1. Global schema** | Runs `prepare_data` once on the full dataset to learn the complete dummy column space |
| **2. Partition** | Shuffles patient IDs, splits into N chunks, writes each client's raw rows to disk |
| **3. Client prep** | Each client independently preprocesses, splits and scales its own data |

The only information shared between stages is `feature_schema.json` — a list of column names with no patient data — which guarantees identical feature dimensionality across all clients.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import joblib
import json
import os

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, classification_report
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import sys
sys.path.append('..')

from UC1Utils import (
    prepare_data, ensure_data,
    prepare_data_aligned, derive_global_columns,
    split_data, scale_data,
    DATA_DIR, CSV_MAIN
)

from UC1FLUtils import ( dirichlet_partition,
                        load_clients,
                        run_fedavg_full,
                        run_fedavg_partial,
                        fl_objective,
                        evaluate_global,
                        save_result
)

DATA_PATH    = CSV_MAIN
OUTPUT_DIR   = '../federated_data'   # shared with FedGen
N_CLIENTS    = 5
ALPHA_SWEEP  = [0.1, 0.5, 1.0, 5.0, 10.0]
SEEDS        = [42, 123, 7]
ALPHA        = ALPHA_SWEEP[0]        # default for single-run inspection
SEED         = SEEDS[0]
MIN_PATIENTS = 500
MIN_POS_RATE = 0.01
MAX_RETRIES  = 500
FILTERED_DIR = 'filtered'
UNFILTERED_DIR = 'unfiltered'
ensure_data()


✓ CSV files already present, skipping extraction.


/mnt/c/Users/Marc/Documents/Uni/4t/TFG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model Setup

In [2]:
# LayerNorm is more stable than BatchNorm for small/heterogeneous FL batches.
# Split into feature_extractor + predictor to support partial sharing variant.
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=2, dropout=0.3):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.LayerNorm(hidden_dim // 4), nn.ReLU(), nn.Dropout(dropout),
        )
        self.predictor = nn.Linear(hidden_dim // 4, output_dim)

    def forward(self, x):
        return self.predictor(self.feature_extractor(x))

    def encode(self, x):
        return self.feature_extractor(x)


In [9]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Tuning setup ──────────────────────────────────────────────────────────────
# Use alpha=1.0, seed=42, client_0 as the tuning client.
# alpha=1.0 is the most balanced partition — representative without being
# pathological. We tune on a single client to preserve the federated assumption
# that no central data access is available during configuration.
_TUNE_ALPHA  = 1.0
_TUNE_SEED   = 42
_TUNE_CLIENT = 0
_TUNE_EPOCHS = 50

_d = os.path.join(OUTPUT_DIR, FILTERED_DIR, f'alpha_{_TUNE_ALPHA}', f'client_{_TUNE_CLIENT}')
_X_tr = torch.tensor(np.load(f'{_d}/X_train.npy'), dtype=torch.float32)
_y_tr = torch.tensor(np.load(f'{_d}/y_train.npy'), dtype=torch.long)
_X_va = torch.tensor(np.load(f'{_d}/X_val.npy'),   dtype=torch.float32)
_y_va = torch.tensor(np.load(f'{_d}/y_val.npy'),   dtype=torch.long)
_input_dim = _X_tr.shape[1]

reoptimal_search = False

# Smallest client train size across all alphas — used to cap batch_size
_min_client_size = min(
    json.load(open(os.path.join(OUTPUT_DIR, FILTERED_DIR, f'alpha_{a}', f'client_{i}',
                                'client_info.json')))['n_train']
    for a in ALPHA_SWEEP for i in range(N_CLIENTS)
)
print(f'Minimum client train size across all partitions: {_min_client_size:,}')
print(f'Tuning on: alpha={_TUNE_ALPHA}, client_{_TUNE_CLIENT}, '
      f'{len(_X_tr):,} train samples')


if reoptimal_search:
    _study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=_TUNE_SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
    )
    _study.optimize(lambda trial: fl_objective(trial, _min_client_size, _input_dim, _X_tr, _y_tr, _X_va, _y_va, _TUNE_EPOCHS, _TUNE_SEED), n_trials=40, show_progress_bar=True)

    # Save for future runs
    _hp_save_path = os.path.join(OUTPUT_DIR, 'fl_hyperparams.json')
    with open(_hp_save_path, 'w') as f:
        json.dump({
            'best_params': _study.best_params,
            'best_value':  _study.best_value,
        }, f, indent=2)
    print(f'Hyperparameters saved → {_hp_save_path}')

else:
    _hp_save_path = os.path.join(OUTPUT_DIR, 'fl_hyperparams.json')
    if not os.path.exists(_hp_save_path):
        raise FileNotFoundError(
            f'No saved hyperparameters at {_hp_save_path}. '
            f'Set reoptimal_search=True to run the search.'
        )
    with open(_hp_save_path) as f:
        _saved = json.load(f)
    class _StudyProxy:
        best_params = {k: v for k, v in _saved.items() if k != 'best_value'}
        best_value  = _saved.get('best_value', float('nan'))
    _study = _StudyProxy()

print(f'\nBest federated AUC (single client, local training): {_study.best_value:.4f}')
print(f'Best params: {_study.best_params}')

Minimum client train size across all partitions: 2,004
Tuning on: alpha=1.0, client_0, 21,064 train samples

Best federated AUC (single client, local training): nan
Best params: {'hidden_dim': 512, 'dropout': 0.18032260117084994, 'lr': 0.009295988540171725, 'batch_size': 256}


In [ ]:
# ── Federated hyperparameters ─────────────────────────────────────────────────
# Sourced from the Optuna search above, tuned on a single client with the
# LayerNorm split architecture. Both FedAvg and FedGen use these values since
# they share the same MLP backbone.

_bp = _study.best_params

FL_ROUNDS    = 30
LOCAL_EPOCHS = 10
HIDDEN_DIM   = _bp['hidden_dim']
DROPOUT      = _bp['dropout']
LR           = _bp['lr']
BATCH_SIZE   = _bp['batch_size']
PATIENCE     = 5
FILTERED_DIR   = '../federated_data/filtered'
UNFILTERED_DIR = '../federated_data/unfiltered'  

print('Federated hyperparameters (tuned on single client, LayerNorm MLP):')
print(f'  hidden_dim : {HIDDEN_DIM}')
print(f'  dropout    : {DROPOUT:.4f}')
print(f'  lr         : {LR:.6f}')
print(f'  batch_size : {BATCH_SIZE}')
print(f'\nFL settings:')
print(f'  rounds     : {FL_ROUNDS}')
print(f'  local_epochs: {LOCAL_EPOCHS}')
print(f'  patience   : {PATIENCE}')

Federated hyperparameters (tuned on single client, LayerNorm MLP):
  hidden_dim : 512
  dropout    : 0.1803
  lr         : 0.009296
  batch_size : 256

FL settings:
  rounds     : 30
  local_epochs: 10
  patience   : 5


## FedAvg

In [13]:
re_train = False
RESULTS_DIR = 'results'

# ── Outer sweep: all alphas × all seeds × both data cases ─────────────────────
for data_case, data_dir in [('filtered', FILTERED_DIR), ('unfiltered', UNFILTERED_DIR)]:
    print(f'\n{"#"*60}')
    print(f'Data case: {data_case}')
    print(f'{"#"*60}')

    for alpha in ALPHA_SWEEP:
        for seed in SEEDS:
            r_dir        = os.path.join(RESULTS_DIR, data_case)
            full_path    = os.path.join(r_dir, f'alpha_{alpha}/seed_{seed}/fedavg_full.json')
            partial_path = os.path.join(r_dir, f'alpha_{alpha}/seed_{seed}/fedavg_partial.json')


            if os.path.exists(full_path) and os.path.exists(partial_path) and not re_train:
                print(f'  {data_case} α={alpha} seed={seed}: already done, skipping.')
                continue

            print(f'\n{"="*60}')
            print(f'FedAvg  [{data_case}]  α={alpha}  seed={seed}')
            print(f'{"="*60}')

            try:
                clients = load_clients(alpha, data_dir, N_CLIENTS)
            except (FileNotFoundError, ValueError) as e:
                print(f'  Skipping α={alpha} [{data_case}]: {e}')
                continue
            input_dim = clients[0]['X_train'].shape[1]

            if not os.path.exists(full_path) or re_train:
                print('\n--- fedavg_full ---')
                auc, pc, hist, cmb = run_fedavg_full(
                    clients, input_dim, seed,
                    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
                    lr=LR, batch_size=BATCH_SIZE
                )
                save_result(r_dir, alpha, seed, 'fedavg_full', auc, pc, hist, cmb)

            if not os.path.exists(partial_path) or re_train:
                print('\n--- fedavg_partial ---')
                auc, pc, hist, cmb = run_fedavg_partial(
                    clients, input_dim, seed,
                    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
                    lr=LR, batch_size=BATCH_SIZE
                )
                save_result(r_dir, alpha, seed, 'fedavg_partial', auc, pc, hist, cmb)

print('\nAll FedAvg experiments complete.')



############################################################
Data case: filtered
############################################################
  filtered α=0.1 seed=42: already done, skipping.
  filtered α=0.1 seed=123: already done, skipping.
  filtered α=0.1 seed=7: already done, skipping.
  filtered α=0.5 seed=42: already done, skipping.
  filtered α=0.5 seed=123: already done, skipping.
  filtered α=0.5 seed=7: already done, skipping.
  filtered α=1.0 seed=42: already done, skipping.
  filtered α=1.0 seed=123: already done, skipping.
  filtered α=1.0 seed=7: already done, skipping.
  filtered α=5.0 seed=42: already done, skipping.
  filtered α=5.0 seed=123: already done, skipping.
  filtered α=5.0 seed=7: already done, skipping.
  filtered α=10.0 seed=42: already done, skipping.
  filtered α=10.0 seed=123: already done, skipping.
  filtered α=10.0 seed=7: already done, skipping.

############################################################
Data case: unfiltered
####################

In [ ]:
import glob
# After
_glob_unfiltered = os.path.join(RESULTS_DIR, 'unfiltered')
for path in sorted(glob.glob(f'{_glob_unfiltered}/alpha_*/seed_*/*.json')):
    r = json.load(open(path))
    parts = path.replace(f'{_glob_unfiltered}/', '').replace('.json', '').split('/')
    alpha_s, seed_s, variant = parts


The results for the unfiltered data case are: 
  α=alpha_0.5, seed=seed_123, variant=fedavg_full
  Test AUC: 0.5832, Rounds: 6
The results for the unfiltered data case are: 
  α=alpha_0.5, seed=seed_123, variant=fedavg_partial
  Test AUC: 0.7597, Rounds: 7
The results for the unfiltered data case are: 
  α=alpha_0.5, seed=seed_42, variant=fedavg_full
  Test AUC: 0.5785, Rounds: 6
The results for the unfiltered data case are: 
  α=alpha_0.5, seed=seed_42, variant=fedavg_partial
  Test AUC: 0.8028, Rounds: 21
The results for the unfiltered data case are: 
  α=alpha_0.5, seed=seed_7, variant=fedavg_full
  Test AUC: 0.5860, Rounds: 10
The results for the unfiltered data case are: 
  α=alpha_0.5, seed=seed_7, variant=fedavg_partial
  Test AUC: 0.7650, Rounds: 8
The results for the unfiltered data case are: 
  α=alpha_1.0, seed=seed_123, variant=fedavg_full
  Test AUC: 0.6071, Rounds: 12
The results for the unfiltered data case are: 
  α=alpha_1.0, seed=seed_123, variant=fedavg_partial
  Tes